### Machine Learning Pipeline

The dataset used for this example is obtained from [UC Irvine Machine Learning Repository](https://archive.ics.uci.edu/dataset/222/bank+marketing).

The second part consists of creating a machine learning model to predict if the client will subscribe (yes/no) a term deposit (variable y).

#### Import Dataset

In [3]:
%pip install pandas joblib scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [4]:
# import libraries
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import f1_score

In [5]:
# read dataset
df = pd.read_csv("/Users/andresrodartee/mlops-and-system-design/session_2/datasets2/bank-full_train_test.csv")

#### Transform Dataset

In [6]:
# balance dataset
def balance_dataset(df: pd.DataFrame) -> pd.DataFrame:
    # Separate the classes
    df_y0 = df[df['y'] == 'no']
    df_y1 = df[df['y'] == 'yes']

    # Find the smaller class size
    min_size = len(df_y1)

    # Randomly sample from each class
    df_y0_balanced = df_y0.sample(n=min_size, random_state=42)

    # Concatenate back together
    df_balanced = pd.concat([df_y0_balanced, df_y1])

    # Shuffle the dataset
    df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

    return df_balanced

df = balance_dataset(df)

In [7]:
# map target
df.loc[df["y"] == "yes", "y"] = 1
df.loc[df["y"] == "no", "y"] = 0
df["y"] = df["y"].astype(int)

In [8]:
# prepare dataset for classification model
class Transformer:
    def __init__(self):
        self.DROP_COLUMNS = []
        self.BINARY_FEATURES = [
            "housing",
            "loan",
            "default",
        ]

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.drop(self.DROP_COLUMNS, axis=1)
        df = self._map_binary_column_to_int(df)
        df = self._map_month_to_int(df)

        return df

    def _map_binary_column_to_int(self, df: pd.DataFrame) -> pd.DataFrame:
        for col in self.BINARY_FEATURES:
            df[col] = df[col].map({"yes": 1, "no": 0})
        return df

    def _map_month_to_int(self, df: pd.DataFrame) -> pd.DataFrame:
        month_mapping = {
            "jan": 1,
            "feb": 2,
            "mar": 3,
            "apr": 4,
            "may": 5,
            "jun": 6,
            "jul": 7,
            "aug": 8,
            "sep": 9,
            "oct": 10,
            "nov": 11,
            "dec": 12,
        }
        df["month"] = df["month"].map(month_mapping)

        return df

In [9]:
# transform the dataset
df = Transformer().transform(df)

In [10]:
# encode categorical variables
ONE_HOT_ENCODE_COLUMNS = [
            "marital",
            "job",
            "education",
            "poutcome",
            "contact",
        ]

encoder = OneHotEncoder(drop='first', sparse_output=False).set_output(transform="pandas")
encoder.fit(df[ONE_HOT_ENCODE_COLUMNS])
encoded_df = encoder.transform(df[ONE_HOT_ENCODE_COLUMNS])
df = df.drop(columns=ONE_HOT_ENCODE_COLUMNS)
df = pd.concat([df, encoded_df], axis=1)

In [11]:
df.head()

,age,default,balance,housing,loan,day,month,duration,campaign,pdays,...,job_unemployed,job_unknown,education_secondary,education_tertiary,education_unknown,poutcome_other,poutcome_success,poutcome_unknown,contact_telephone,contact_unknown
0,43,0,50,0,0,28,5,90,8,-1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
1,32,0,40,0,0,8,8,419,2,-1,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,32,0,2559,0,0,7,7,889,1,-1,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
3,23,0,480,0,0,9,2,742,2,182,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,32,0,1169,1,0,21,5,322,2,-1,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0


In [12]:
# save encoder - future use
joblib.dump(encoder, 'model_utils/encoder.pkl')

['model_utils/encoder.pkl']

#### Train and evaluate model

In [13]:
# Split the data into training and test sets
X = df.drop('y', axis=1)
y = df['y']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [14]:
# Define the model hyperparameters
params = {
    "solver": "lbfgs",
    "max_iter": 1000,
    "multi_class": "auto",
    "random_state": 8888,
}

In [15]:
# Train the model
lr = LogisticRegression(**params)
lr.fit(X_train, y_train)

/Users/andresrodartee/mlops-and-system-design/.venv-1/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/andresrodartee/mlops-and-system-design/.venv-1/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=1000, multi_class='auto', random_state=8888)

In [16]:
# Predict on the test set
y_pred = lr.predict(X_test)

In [17]:
# Calculate metrics
f1_score = f1_score(y_test, y_pred)

### MLFlow Integration

In [18]:
import sys
!{sys.executable} -m pip install mlflow

In [19]:
# import libraries
import mlflow
from mlflow.models import infer_signature

/Users/andresrodartee/mlops-and-system-design/.venv-1/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In a new terminal, run a local Tracking Server with the following command:

`mlflow server --host 127.0.0.1 --port 8080`

In [20]:
# Set our tracking server uri for logging
mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")

In [21]:
# Create a new MLflow Experiment
mlflow.set_experiment("LR Experiment")

# Start an MLflow run
with mlflow.start_run():
    # Log the hyperparameters
    mlflow.log_params(params)

    # Log the loss metric
    mlflow.log_metric("f1_score", f1_score)

    # Set a tag that we can use to remind ourselves what this run was for
    mlflow.set_tag("Training Info", "Basic LR model to present MLFlow capabilities")

    # Infer the model signature
    signature = infer_signature(X_train, lr.predict(X_train))

    # Log the model
    model_info = mlflow.sklearn.log_model(
        sk_model=lr,
        artifact_path="bank_model",
        signature=signature,
        input_example=X_train,
        registered_model_name="bank-model-basic",
    )

/Users/andresrodartee/mlops-and-system-design/.venv-1/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/14 19:48:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Registered model 'bank-model-basic' already exists. Creating a new version of this model...
2026/05

🏃 View run unruly-hound-127 at: http://127.0.0.1:8080/#/experiments/649876885013562386/runs/ebee725dea3242afa013aaf8d5b1bb40
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/649876885013562386


Created version '2' of model 'bank-model-basic'.
